In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/karkavelrajaj/amazon-sales-dataset/amazon.csv


# Amazon Sales Analysis Report 

# Executive Summary
This report analyzes sales performance, pricing strategies, and customer sentiment based on the Amazon Sales dataset. The data reveals that the Fire-Boltt Ninja Call Pro Plus is the most frequent product. While most sales occur at lower discount rates (under 20%), customer satisfaction remains high across the board, particularly in the Computers & Accessories category.


# Objective
The primary objective of this analysis is to identify patterns between product attributes (price, discounts, ratings) and customer feedback to understand what drives volume and satisfaction on the platform.


# Data Cleaning & Preprocessing
To ensure the integrity of the findings, the following data cleaning steps were performed:

**Currency Standardization**: Removed currency symbols (e.g., ₹) and commas from discounted_price and actual_price to convert them into numeric formats.

**Handling Missing Values**: Identified and addressed null entries in rating_count and product descriptions to maintain statistical accuracy.

**Data Type Conversion**: Converted the rating column to a float; specifically handled non-numeric characters (like "heavy" or "invalid" strings) found in raw rating data.

**Percentage Formatting**: Stripped the '%' sign from discount_percentage to enable mathematical correlation analysis.

**Category Splitting:** Segregated the main category from sub-categories (e.g., "Computers & Accessories | Tablets") to perform granular category-wise analysis.



# Main Findings

**Sales & Discount Trends**

**High Volume at Low Discounts**: Most transactions occur at full price or with small discounts (under 20%).

**Fewer High-Discount Sales**: Deep discounts (40–80%) make up a significantly smaller portion of total transactions.

**Product Dominance**: The Fire-Boltt Ninja Call Pro Plus is the most frequently occurring product in the dataset.


**Customer Sentiment & Ratings**

**Dominant Positivity**: A "dominance of green" across sentiment charts indicates that the majority of products receive positive feedback.

**Top Performer**: The Computers & Accessories | Tablets category holds the highest average rating.

**Sentiment Density**: Many products share a similar, high number of reviews, with most categories maintaining an average rating above 3.50.

**Areas for Improvement**: Certain categories (notably segments 2 and 3) show larger "red segments" (negative feedback), suggesting potential quality or service issues.


**Statistical Correlations**

**Price vs. Rating**: Ratings do not reliably predict a product's price. For example, products rated 4.0 vary wildly in price, from budget items to those costing ₹70,000.


# Key Relationships:

**Actual vs. Discounted Price**: Strong positive correlation (prices move together).

**Rating vs. Rating Count**: Positive correlation, suggesting products with more reviews tend to have slightly higher average ratings.


**Discount vs. Price**: Negative correlation; higher percentage discounts are typically applied to lower-priced products.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("/kaggle/input/amazon-sales-dataset/amazon.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/amazon-sales-dataset/amazon.csv'

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Changing the data type of columns and also removing prefixes and suffixes
df['discounted_price'] = df['discounted_price'].str.replace('₹', '').str.replace(',', '')
df['discounted_price'] = df['discounted_price'].astype('float64')
df['actual_price'] = df['actual_price'].str.replace('₹', '').str.replace(',', '')
df['actual_price'] = df['actual_price'].astype('float64')
df['discount_percentage'] = df['discount_percentage'].str.replace('%', '')

df['discount_percentage'] = df['discount_percentage'].astype('float64')

In [ ]:
# Changing the data of Discount% column
df['discount_percentage'] = df['discount_percentage']/100
df['discount_percentage'] 

In [ ]:
# Replaing null Value in the column
df['rating_count'] = df['rating_count'].str.replace("," , "")
df['rating_count']= df['rating_count'].astype("float64")

In [ ]:
# Find rows that cannot be converted to numeric
non_numeric_rows = df[pd.to_numeric(df['rating'], errors='coerce').isnull()]

print(non_numeric_rows)

In [ ]:
df['rating'] = df['rating'].replace('|', 3.9).astype(float)

In [ ]:
value= df.iloc[1279,6]
value

In [ ]:
# Find Duplicate
df.duplicated().any()
# No dulpicates in te dataset

In [ ]:
df.nunique()

In [ ]:
product_stats = df.groupby('product_name').agg({
    'rating': 'mean',
    'rating_count': 'sum' # Assuming count represents total sales or engagements
})
product_stats
best_products = product_stats.sort_values(
    by=['rating', 'rating_count'], ascending=False
).head(10)
best_products

In [ ]:
# 4. Plotting
best_products['rating'].plot(kind='barh', figsize=(10, 6), color='skyblue')
plt.title('Best Selling Products by Rating & Volume')
plt.xlabel('Average Rating')
plt.ylabel('Product Name')
plt.gca().invert_yaxis() # Highest rating on top
plt.show()
# Amazon Basics Wireless Mouse got max rating and rating count

In [ ]:
# Count occurrences of product names
product_counts = df["product_name"].value_counts()

# Sort in descending order and display top results
print(product_counts.sort_values(ascending=False).head(10))

In [ ]:
# 5. Scatter plot for Price vs Discount
plt.figure(figsize=(10, 6))
sns.scatterplot(x='discount_percentage', y='discounted_price', data=df, alpha=0.5)
plt.title('Discounted Price vs Discount Percentage')
plt.show()

The data suggest High volume at low discounts: Sales occur at full price or with discount(under 20)

Fewer high discout sales: Discount of (40-80)% make up a smaller transaction

In [ ]:
# 2. Define a simple custom lexicon (rule-based approach)
positive_words = set(['love', 'great', 'works', 'perfectly', 'best', 'awesome', 'highly', 'recommend', 'good', 'happy'])
negative_words = set(['not', 'bad', 'broke', 'disappointed', 'terrible', 'worst', 'hate', 'upset', 'problem', 'slow'])

# 3. Function to calculate a simple sentiment score
def simple_sentiment(text):
    if isinstance(text, str):
        words = text.lower().split()
        score = sum(1 for word in words if word in positive_words) - sum(1 for word in words if word in negative_words)
        return score
    return 0

# 4. Apply the sentiment analysis to a new column
# Combine title and content for a more comprehensive analysis
df['full_review'] = df['review_title'] + " " + df['review_content']
df['sentiment_score'] = df['full_review'].apply(simple_sentiment)

# 5. Classify sentiment into categories (Positive, Negative, Neutral)
def classify_sentiment(score):
    if score > 0:
        return 'Positive'
    elif score < 0:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_type'] = df['sentiment_score'].apply(classify_sentiment)

# 6. Analyze sentiment by product and plot using pandas and matplotlib
# Group by product_id and sentiment_type to count occurrences
sentiment_counts = df.groupby(['product_id', 'sentiment_type']).size().unstack(fill_value=0)

# Plotting the results using pandas' built-in plot function which uses matplotlib
sentiment_counts.plot(kind='bar', stacked=True, figsize=(10, 6), color=['red', 'gray', 'green'])
plt.title('Customer Sentiment Distribution per Product')
plt.xlabel('Product ID')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)
plt.legend(title='Sentiment')
plt.show()

**For each product bar, the coloured segmet show the proportion of review falling in to each sentiment category. Some have largest red segment while others have evev distribution or a larger negative segment. Bars are dense at left side, suggesting mny products have a similar , high numbe rof of reviews. There is a dominance of green across the enire chart suggests that the majority of produts have received positive feedback**

In [ ]:
# Calculate the average ratings after ensuring numeric data type
average_ratings = df.groupby("category")["rating"].mean().sort_values(ascending = False).reset_index()

print(average_ratings.head(5))

Average rating for each product category: Computers&Accessories|Tablets has the highest average rating The output shows that most product categories have generally positive customer feedback, with average ratings above 3.50. However, some categories (e.g., 2 and 3) have lower ratings, suggesting potential areas for improvement. Further analysis of these categories could help identify specific reasons for lower feedback and identify potential solutions.

In [ ]:
# Calculate the average rating per category
average_ratings = df.groupby('category')['rating'].mean().sort_values(ascending=False)

# Select the top 5 categories
top_5_categories = average_ratings.head(5)

# Plotting the results
plt.figure(figsize=(10, 6))
sns.barplot(x=top_5_categories.values, y=top_5_categories.index, palette='viridis')
plt.title('Top 5 Product Categories by Average Rating')
plt.xlabel('Average Rating')
plt.ylabel('Category')
plt.xlim(min(top_5_categories.values) - 0.1, 5.0) # Ratings are typically out of 5
plt.show()


**Computer and Accessories** Tablts recently received highest no of rating

In [ ]:
# Plotting rating vs discounted price
plt.figure(figsize=(10, 6))
sns.scatterplot(x='rating', y='discounted_price', data=df, alpha=0.5)
plt.title('Effect of Rating on Discounted Price')
plt.xlabel('Rating')
plt.ylabel('Discounted Price (INR)')
plt.show()

For any given rating level, there is a wide range of discounted price e.g. prapprox 7000oduct rated around 4.0 vary in price from very low amounts upto 70000 Higher or lower rating daoesnot reliably predict a product's discounted price based on this dataset

In [ ]:
# Top 5 most frequent reviews within each category
top_categories = df['category'].value_counts().head(5)
print("Top 5 Categories by Review Count:")
print(top_categories)

In [ ]:
#  Select Numerical Features for Correlation ---
# Focusing on Price, Discount, and Rating relationships
numerical_data = df[['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']]

#  Generate Correlation Matrix ---
corr_matrix = numerical_data.corr()

# --- 4. Plot Heatmap ---
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, 
            annot=True,         # Show numbers in cells
            cmap='coolwarm',    # Color palette
            fmt='.2f',          # Format numbers to 2 decimal places
            linewidths=0.5)     # Add lines between cells

plt.title('Amazon Product Attribute Correlation Heatmap')
plt.show()

Relationship among Amazon Product attributes: actual_price and discounted_price: Strong positive corelation which is expected as the discounted price is derived from the actual price rating and rating_count: Positive correlation suggesting a slight tendency for product with more ratings to have higher average ratings discount_percentage and prices: Negative correlation suggesting that higher discunt percentage might be applied to products with slightly lower prices rating_count and prices: Slightly positive correlation implying that more expensive items might receive slightly more ratings

In [ ]:
# Setup the visualization dashboard layout
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.subplots_adjust(hspace=0.3, wspace=0.2)

# 1. Actual vs. Discounted Price (Strong Positive Correlation)
sns.regplot(x='actual_price', y='discounted_price', data=df, ax=axes[0, 0], color='teal')
axes[0, 0].set_title('Actual vs. Discounted Price (Strong Correlation)', fontsize=14)

# 2. Price vs. Rating (Low Correlation / High Variance)
# Shows that products with a 4.0 rating vary wildly in price
sns.scatterplot(x='rating', y='actual_price', data=df, ax=axes[0, 1], color='coral', alpha=0.6)
axes[0, 1].set_title('Price vs. Rating (High Variance)', fontsize=14)

# 3. Discount % vs. Actual Price (Negative Correlation)
# Higher discounts typically applied to lower-priced items
sns.regplot(x='actual_price', y='discount_percentage', data=df, ax=axes[1, 0], color='purple')
axes[1, 0].set_title('Discount % vs. Price (Negative Correlation)', fontsize=14)

# 4. Rating vs. Rating Count (Social Proof Effect)
# Higher review counts generally correlate with slightly higher ratings
sns.regplot(x='rating', y='rating_count', data=df, ax=axes[1, 1], color='forestgreen')
axes[1, 1].set_title('Rating vs. Rating Count (Social Proof)', fontsize=14)

plt.show()

# Conclusion
The analysis shows a market characterized by high-volume, low-discount sales with predominantly positive customer feedback. While specific product categories like Tablets perform exceptionally well, there are opportunities to improve customer sentiment in specific sub-segments. Price and rating are not strongly related, indicating a competitive landscape where value is subjective to the buyer.